# 1. Library Import


In [ ]:
!pip install ultralytics

In [ ]:
# ==========================================
# 1. Standard Libraries (ফাইল এবং ডিরেক্টরি ম্যানেজমেন্ট)
# ==========================================
import os
import glob
import json
import shutil
import xml.etree.ElementTree as ET # XML পার্স করার জন্য (যদি লেবেল XML এ থাকে)

# ==========================================
# 2. Data Processing & Math
# ==========================================
import numpy as np
import pandas as pd

# ==========================================
# 3. Image Processing (OpenCV ও অগমেন্টেশন)
# ==========================================
import cv2
from PIL import Image

# ==========================================
# 4. Deep Learning Framework & YOLO
# ==========================================
import torch
import torchvision
from ultralytics import YOLO # YOLOv8 বা অন্যান্য ভার্সনের জন্য

# ==========================================
# 5. Evaluation Metrics & Validation (k-fold, F1, Recall)
# ==========================================
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, roc_curve, auc

# ==========================================
# 6. Visualization (গ্রাফ এবং চার্ট)
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 7. Device Configuration (GPU চেক করা)
# ==========================================
# Kaggle-এ GPU ঠিকমতো পাচ্ছে কি না, তা নিশ্চিত করার জন্য
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")

# 2. Data Pre-Processing 

In [ ]:

# Define base path (according to your output)
dataset_base_path = '/kaggle/input/datasets/ariadnapartinen/cadica-a-new-dataset-for-coronary-artery-disease/CADICA a new dataset for coronary artery disease/CADICA/CADICA/selectedVideos'
processed_images_dir = '/kaggle/working/processed_cadica/images'
processed_labels_dir = '/kaggle/working/processed_cadica/labels'

# Create folders (if not already created)
os.makedirs(processed_images_dir, exist_ok=True)
os.makedirs(processed_labels_dir, exist_ok=True)

# CLAHE and Resize function (written previously)
def apply_clahe_and_resize(image_path, save_path, size=(640, 640)):
    img = cv2.imread(image_path)
    if img is None: return False
    
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    
    limg = cv2.merge((cl, a, b))
    final_img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    resized_img = cv2.resize(final_img, size)
    
    cv2.imwrite(save_path, resized_img)
    return True

print("🔄 Dataset processing is starting... it might take some time...")

total_processed = 0
# Traverse the entire directory tree using os.walk
for root, dirs, files in os.walk(dataset_base_path):
    # We are only looking for the 'input' folder, because that's where the images are
    if root.endswith('input'):
        for file in files:
            if file.endswith('.png'): # Images are in .png format
                image_path = os.path.join(root, file)
                
                # Create label file name matching the image name
                # Example: p18_v10_00035.png -> p18_v10_00035.txt
                base_name = os.path.splitext(file)[0]
                label_filename = base_name + '.txt'
                
                # The label file is usually located in the groundtruth folder parallel to the input folder
                parent_dir = os.path.dirname(root) # Folder above input (e.g., v10)
                label_path = os.path.join(parent_dir, 'groundtruth', label_filename)
                
                # We will process the image only if the label file exists
                if os.path.exists(label_path):
                    # Process and save the image
                    save_img_path = os.path.join(processed_images_dir, file)
                    apply_clahe_and_resize(image_path, save_img_path)
                    
                    # Copy the label file directly
                    save_label_path = os.path.join(processed_labels_dir, label_filename)
                    shutil.copy2(label_path, save_label_path)
                    
                    total_processed += 1

print(f"✅ Processing complete! A total of {total_processed} pairs of images and labels have been collected.")

# 2.1 Output Not Provide YOLO Format

In [ ]:


labels_dir = '/kaggle/working/processed_cadica/labels'
# যেকোনো একটি লেবেল ফাইল সিলেক্ট করা
sample_label = os.listdir(labels_dir)[0]

print(f"Sample Label File: {sample_label}")
print("--- Content ---")
with open(os.path.join(labels_dir, sample_label), 'r') as f:
    print(f.read())

# 2.2 Data Formating to YOLO Format

In [ ]:
labels_dir = '/kaggle/working/processed_cadica/labels'
original_base_path = '/kaggle/input/datasets/ariadnapartinen/cadica-a-new-dataset-for-coronary-artery-disease/CADICA a new dataset for coronary artery disease/CADICA/CADICA/selectedVideos'

# 1. Creating a dictionary to quickly find original images
print("Finding the size of original images...")
original_images = {}
for root, dirs, files in os.walk(original_base_path):
    if root.endswith('input'):
        for file in files:
            if file.endswith('.png'):
                original_images[file] = os.path.join(root, file)

# 2. Converting labels to YOLO format
print("Converting labels to YOLO format...")
label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
processed_count = 0

for label_file in label_files:
    image_filename = label_file.replace('.txt', '.png')
    
    # Skip if the original image is not found
    if image_filename not in original_images:
        continue 
        
    # Extracting Width and Height of the original image
    orig_img = cv2.imread(original_images[image_filename])
    if orig_img is None: continue
    orig_h, orig_w = orig_img.shape[:2]
    
    label_path = os.path.join(labels_dir, label_file)
    with open(label_path, 'r') as f:
        lines = f.readlines()
        
    yolo_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 4:
            try:
                x_min = float(parts[0])
                y_min = float(parts[1])
                w = float(parts[2])
                h = float(parts[3])
                
                # Calculation for YOLO format (between 0 and 1)
                x_center = (x_min + w / 2) / orig_w
                y_center = (y_min + h / 2) / orig_h
                w_norm = w / orig_w
                h_norm = h / orig_h
                
                # Class ID = 0 (Since we are only detecting CAD)
                yolo_lines.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
            except ValueError:
                continue
                
    # Overwriting the old label file with the converted values
    with open(label_path, 'w') as f:
        f.write('\n'.join(yolo_lines))
        
    processed_count += 1

print(f"✅ Conversion successful! A total of {processed_count} label files are ready in YOLO format.")

# Printing a label for verification
sample_label = label_files[0]
print(f"\nSample YOLO Label File ({sample_label}):")
with open(os.path.join(labels_dir, sample_label), 'r') as f:
    print(f.read())

# 3. K-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import KFold
import os # Added os import just in case, though it might be above your snippet

# Creating the full path of the images
images_dir = '/kaggle/working/processed_cadica/images'
all_images = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith('.png')])

# K-Fold setup (k=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Creating folder to save split data
fold_dir = '/kaggle/working/yolo_kfold_splits'
os.makedirs(fold_dir, exist_ok=True)

print("K-Fold splitting is starting...\n")

# Loop for 5 folds
for fold, (train_idx, val_idx) in enumerate(kf.split(all_images), 1):
    
    # Separating train and validation data paths according to index
    train_files = [all_images[i] for i in train_idx]
    val_files = [all_images[i] for i in val_idx]
    
    # Saving paths in text files
    train_txt = os.path.join(fold_dir, f'fold_{fold}_train.txt')
    val_txt = os.path.join(fold_dir, f'fold_{fold}_val.txt')
    
    with open(train_txt, 'w') as f:
        f.write('\n'.join(train_files))
        
    with open(val_txt, 'w') as f:
        f.write('\n'.join(val_files))
        
    # Creating data.yaml for this specific fold
    yaml_content = f"""
train: {train_txt}
val: {val_txt}

nc: 1
names: ['CAD']
"""
    yaml_file = os.path.join(fold_dir, f'data_fold_{fold}.yaml')
    with open(yaml_file, 'w') as f:
        f.write(yaml_content.strip())
        
    print(f"Fold {fold}: Train={len(train_files)} images, Val={len(val_files)} images | YAML saved as data_fold_{fold}.yaml")

print("\n✅ 5-Fold Cross Validation split completed successfully!")

# 4. YOLOv11 Model Train

In [ ]:
import time
from ultralytics import YOLO
import gc

# Message will be printed at the very beginning
print("*"*60, flush=True)
print("⏳ YOLO11 Complete 5-Fold Cross-Validation is starting...", flush=True)
print("*"*60, flush=True)

time.sleep(2) 

# The loop will run from 1 to 5 (1, 2, 3, 4, 5)
for fold in range(1, 6):
    print("\n" + "="*60, flush=True)
    print(f"🚀 Training Fold {fold} / 5 is starting...", flush=True)
    print("="*60, flush=True)
    
    # Freeing up memory (so GPU doesn't overload)
    gc.collect()
    torch.cuda.empty_cache()
    
    # YOLO11 এর নতুন মডেল লোড করা (এখানে Nano মডেল yolo11n.pt ব্যবহার করা হয়েছে)
    model = YOLO('yolo11n.pt') 
    
    # Model training
    results = model.train(
        data=f'/kaggle/working/yolo_kfold_splits/data_fold_{fold}.yaml',
        epochs=50,             
        imgsz=640,             
        batch=16,               
        patience=15,           
        save=True,             
        project='/kaggle/working/yolo11_training', # প্রজেক্টের নাম পরিবর্তন করা হয়েছে
        name=f'yolo11_fold_{fold}_main',           # ফোল্ডারের নাম পরিবর্তন করা হয়েছে 
        device=0,
        verbose=True,      # Will show training updates
        workers=4          # Dataloader fix
    )
    
    print(f"\n✅ Training of Fold {fold} completed successfully!\n", flush=True)

print("🎉 Congratulations! Training for all 5 folds has been completed!", flush=True)

# 5. Training Performace

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import glob

# ছবিগুলোকে নিচে নিচে দেখানোর জন্য ফাংশন
def show_images_vertically(image_pattern, title):
    fig, axes = plt.subplots(5, 1, figsize=(10, 25))
    fig.suptitle(title, fontsize=20, fontweight='bold', y=1.01)
    
    for fold in range(1, 6):
        ax = axes[fold-1]
        
        # এখানে পাথটি আপডেট করা হয়েছে (yolo11_training এবং yolo11_fold)
        result_dir = f'/kaggle/working/yolo11_training/yolo11_fold_{fold}_main'
        
        # Pattern অনুযায়ী image file খোঁজা
        files = glob.glob(f'{result_dir}/{image_pattern}')
        
        if files and os.path.exists(files[0]):
            img = mpimg.imread(files[0])
            _ = ax.imshow(img)
            _ = ax.set_title(f'Fold {fold}', fontsize=16)
            _ = ax.axis('off')
        else:
            _ = ax.text(0.5, 0.5, 'Image Not Found', ha='center', va='center', fontsize=12)
            _ = ax.set_title(f'Fold {fold} (Files not found at {result_dir})', fontsize=12)
            _ = ax.axis('off')
            
    plt.tight_layout()
    plt.show()

# গ্রাফগুলো দেখার জন্য নিচের লাইনগুলো রান করুন
print("🌟 Showing Combined Visual Performance of 5 Folds (YOLO11) 🌟\n")

# 1. F1-Score Curves
show_images_vertically('*F1*.png', '📈 F1-Score Curves (Fold 1 to 5)')

# 2. Precision-Recall (PR) Curves
show_images_vertically('*PR*.png', '📉 Precision-Recall (PR) Curves (Fold 1 to 5)')

# 3. Training Results (Loss & mAP)
show_images_vertically('results.png', '📊 Training Results [Loss & mAP] (Fold 1 to 5)')

# 4. Confusion Matrices
show_images_vertically('confusion_matrix.png', '🧩 Confusion Matrices (Fold 1 to 5)')

# 5. Validation Predictions (Batch 0)
show_images_vertically('val_batch0_pred.jpg', '🖼️ Validation Predictions (Fold 1 to 5)')

print("\n✅ All graphical results merged and displayed successfully!")

# 5.1 Average Result

In [ ]:
import pandas as pd
import os

print("🌟 5-Fold Cross-Validation: Overall Average Performance (YOLOv11) 🌟\n")

metrics_list = []

# Loop through 1 to 5 folds
for fold in range(1, 6):
    # পাথটি YOLOv11 অনুযায়ী আপডেট করা হয়েছে
    csv_path = f'/kaggle/working/yolo11_training/yolo11_fold_{fold}_main/results.csv'
    
    if os.path.exists(csv_path):
        # Reading the CSV file
        df = pd.read_csv(csv_path)
        # Removing spaces from column names
        df.columns = df.columns.str.strip()
        
        # YOLOv11 এর জন্য কলামের নামগুলো সাধারণত এমন হয়:
        map_col = 'metrics/mAP50(B)'
        p_col = 'metrics/precision(B)'
        r_col = 'metrics/recall(B)'
        
        # যদি এই নামগুলো না পায়, তবে একটি চেক রাখা ভালো
        if map_col not in df.columns:
            map_col = 'metrics/mAP50'
            p_col = 'metrics/precision'
            r_col = 'metrics/recall'
        
        # Extracting data for the epoch where mAP was highest
        best_idx = df[map_col].idxmax()
        best_row = df.iloc[best_idx]
        
        metrics_list.append({
            'Fold': f"Fold {fold}", 
            'Precision': best_row[p_col], 
            'Recall': best_row[r_col], 
            'mAP50': best_row[map_col]
        })
    else:
        print(f"❌ results.csv for Fold {fold} not found at {csv_path}")

# If data for at least one fold is found
if metrics_list:
    results_df = pd.DataFrame(metrics_list)
    
    # 🎯 Calculating the Average
    avg_p = results_df['Precision'].mean()
    avg_r = results_df['Recall'].mean()
    avg_map50 = results_df['mAP50'].mean()
    
    print("📊 Best performance of each fold:")
    print("-" * 50)
    print(results_df.to_string(index=False))
    print("-" * 50)
    
    
    print("    FINAL OVERALL PERFORMANCE (Average of 5 Folds)    ")
    print("-" * 50)
    print(f"🔹 Average Precision : {avg_p:.4f}  ({avg_p*100:.2f}%)")
    print(f"🔹 Average Recall    : {avg_r:.4f}  ({avg_r*100:.2f}%)")
    print(f"🔹 Average mAP@50    : {avg_map50:.4f}  ({avg_map50*100:.2f}%)")
    

# 6. Test Image

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os
import cv2

# ১. ৫টি ফোল্ডারের মডেল পাথ সেটআপ
# নিশ্চিত করুন যে এই পাথগুলো আপনার ট্রেনিং আউটপুটের সাথে মিলেছে
model_paths = [f'/kaggle/working/yolo11_training/yolo11_fold_{i}_main/weights/best.pt' for i in range(1, 6)]

# মডেলগুলো লোড করা
print("🚀 Loading all 5 models...")
models = [YOLO(path) for path in model_paths]

# ২. টেস্ট ইমেজের পাথ (এখানে আপনার টেস্ট ইমেজটি দিন)
image_path = '/kaggle/input/datasets/mdhasinaljubaier/test-cad/test_image_1.jpg'

# ৩. ৫টি মডেল দিয়ে প্রেডিকশন চালানো
print(f"🔍 Starting detection on: {os.path.basename(image_path)}")
results_list = [model.predict(source=image_path, conf=0.3, imgsz=640, verbose=False) for model in models]

# ৪. ভিজ্যুয়ালাইজেশন (রেজাল্টগুলো গ্রিডে দেখানো)
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle(f'Detection Comparison Across 5 Folds', fontsize=20, fontweight='bold')

for i, res in enumerate(results_list):
    # প্রেডিকশন ইমেজটি প্লট করা
    plot_img = res[0].plot()  # YOLO-এর বিল্ট-ইন প্লট ফাংশন
    
    ax = axes[i]
    ax.imshow(plot_img[:, :, ::-1]) # BGR থেকে RGB তে কনভার্ট
    ax.set_title(f'Fold {i+1} Prediction', fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n✅ All folds prediction completed and displayed successfully!")